# Day 019 Project Solution — Eval Harness

In [ ]:
import re, ollama
from dataclasses import dataclass, field

def exact_match(response: str, expected: str) -> bool:
    return response.strip().lower() == expected.strip().lower()

def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)

JUDGE_PROMPT = """\
You are an evaluation judge. Score the response below on a scale of 1 to 5.

Question: {question}
Expected answer: {expected}
Actual response: {response}

Rubric:
1 = Completely wrong or irrelevant
2 = Mostly wrong with minor correct elements
3 = Partially correct but with significant gaps
4 = Mostly correct with minor issues
5 = Fully correct and complete

Respond with ONLY this format:
Score: <1-5>
Rationale: <one sentence>
"""

def llm_judge(question, response, expected, model='llama3.2'):
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, response=response)
    raw = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    text = raw['message']['content']
    m = re.search(r'Score:\s*([1-5])', text)
    score = int(m.group(1)) if m else 3
    r = re.search(r'Rationale:\s*(.+)', text)
    rationale = r.group(1).strip() if r else text.strip()[:200]
    return {'score': score, 'rationale': rationale}

@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)
    expected_answer: str = ''

@dataclass
class EvalResult:
    test_case: TestCase
    response: str
    passed: bool
    matched_keywords: list[str] = field(default_factory=list)
    judge_score: int = 0
    judge_rationale: str = ''

def run_eval(test_cases, system_prompt='You are a helpful assistant.',
             model='llama3.2', use_judge=False):
    results = []
    for tc in test_cases:
        raw = ollama.chat(model=model, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': tc.question},
        ])
        response = raw['message']['content']
        matched = [kw for kw in tc.expected_keywords
                   if kw.lower() in response.lower()]
        passed = bool(matched) if tc.expected_keywords else True
        result = EvalResult(tc, response, passed, matched)
        if use_judge and tc.expected_answer:
            j = llm_judge(tc.question, response, tc.expected_answer, model)
            result.judge_score = j['score']
            result.judge_rationale = j['rationale']
        results.append(result)
    return results

def summarize_results(results):
    if not results:
        return {'total': 0, 'passed': 0, 'failed': 0,
                'pass_rate': 0.0, 'avg_judge_score': 0.0}
    total  = len(results)
    passed_count = sum(1 for r in results if r.passed)
    scores = [r.judge_score for r in results if r.judge_score > 0]
    return {
        'total': total,
        'passed': passed_count,
        'failed': total - passed_count,
        'pass_rate': round(passed_count / total, 4),
        'avg_judge_score': round(sum(scores) / len(scores), 2) if scores else 0.0,
    }

def print_report(results):
    s = summarize_results(results)
    print(f"Eval — {s['total']} cases | "
          f"Pass: {s['passed']}/{s['total']} ({s['pass_rate']*100:.1f}%)")
    if s['avg_judge_score'] > 0:
        print(f"  Avg judge score: {s['avg_judge_score']:.1f}/5")
    for i, r in enumerate(results, 1):
        icon = '✅' if r.passed else '❌'
        print(f"  {icon} {i}. {r.test_case.question[:55]}")
        if r.matched_keywords:
            print(f"     Keywords: {r.matched_keywords}")
        if r.judge_score:
            print(f"     Score {r.judge_score}/5 — {r.judge_rationale[:60]}")

SYSTEM_PROMPT = 'You are a helpful and accurate assistant. Answer concisely.'

EVAL_CASES = [
    TestCase('What is the capital of France?',
             expected_keywords=['paris'], expected_answer='Paris'),
    TestCase('What is the chemical symbol for water?',
             expected_keywords=['h2o', 'h₂o'], expected_answer='H2O'),
    TestCase('In what year did World War II end?',
             expected_keywords=['1945'], expected_answer='1945'),
    TestCase('What does len() return in Python?',
             expected_keywords=['length', 'number', 'count', 'size'],
             expected_answer='The number of items in an object'),
    TestCase('Name a planet in our solar system.',
             expected_keywords=['mercury', 'venus', 'earth', 'mars',
                                'jupiter', 'saturn', 'uranus', 'neptune']),
]

results = run_eval(EVAL_CASES, system_prompt=SYSTEM_PROMPT)
print_report(results)
print()
s = summarize_results(results)
print(f"Pass rate: {s['pass_rate']*100:.1f}%")
